In [ ]:
pip install opencv-python torch twilio firebase-admin cloudinary pyserial

Import Liabraries

In [ ]:
import cv2
import time
import torch
from ultralytics import YOLO
from twilio.rest import Client
import firebase_admin
from firebase_admin import credentials, db
import cloudinary
import cloudinary.uploader
from datetime import datetime
import os
import threading
import serial

os.makedirs("Detected_images", exist_ok=True)
os.makedirs("chunk_video", exist_ok=True)
os.makedirs("local_videos", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Libraries imported successfully!")

Importing YOLO model,Twilio, Firebase and Cloudinary

In [ ]:
model = YOLO("runs/detect/train/weights/best.pt").to(device)

account_sid = "ACb7e68017d8c0e7222c1ebf5d711d5f48"
auth_token = "e049ce34b79b4328911c720cf3bf5e7f"
twilio_phone = "whatsapp:+14155238886"
security_phone = "whatsapp:+94760888823"
client = Client(account_sid, auth_token)

cred = credentials.Certificate('Credentials.json')
if not firebase_admin._apps:
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://secuvision-8b0d0-default-rtdb.asia-southeast1.firebasedatabase.app/'
    })
db_ref = db.reference('shoplifting_detections')

cloudinary.config(
    cloud_name="dh7l9ticm",
    api_key="368642485389973",
    api_secret="nHpn21i6cPEDQWSBwKkqqUB0fRU"
)
print("Initialized YOLO, Twilio, Firebase, Cloudinary")

Setting up Arduino

In [ ]:
serial_port = "COM3"
baud_rate = 9600
try:
    ser = serial.Serial(serial_port, baud_rate, timeout=1)
    time.sleep(2)
    print(f"Connected to Arduino on {serial_port}")
except serial.SerialException as e:
    print(f"Could not connect to Arduino: {e}")
    ser = None

Importing the video Source

In [ ]:
def initialize_video_source(source):
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f"Error: Could not open video source {source}.")
        return None
    print(f"Video source {source} opened successfully!")
    return cap

video_source = input("Enter video source (IP webcam URL like 'http://<ip>:8080/video' or local file path, e.g., 'test_video/1.mp4'): ") or "http://192.168.1.100:8080/video"
cap = initialize_video_source(video_source)
if cap is None:
    exit()

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
chunk_duration = 10
fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 1280
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 720

Processing Video and using threading

In [ ]:
confidence_threshold = 0.7
alert_cooldown = 5
last_alert_time = 0
frame_skip = 2
frame_count = 0

def upload_to_cloudinary(chunk_filename):
    try:
        file_size = os.path.getsize(chunk_filename) / (1024 * 1024)
        print(f"Starting upload of {chunk_filename} ({file_size:.2f} MB)")
        upload_result = cloudinary.uploader.upload(
            chunk_filename,
            resource_type="video",
            public_id=f"detected_clips/{os.path.basename(chunk_filename)}",
            overwrite=True
        )
        print(f"Uploaded to Cloudinary: {upload_result['secure_url']}")
    except Exception as e:
        print(f"Cloudinary upload error for {chunk_filename}: {e}")
    finally:
        if os.path.exists(chunk_filename):
            os.remove(chunk_filename)
            print(f"Deleted local chunk file: {chunk_filename}")

def save_local_video(chunk_filename):
    try:
        local_path = os.path.join("local_videos", os.path.basename(chunk_filename))
        shutil.copy2(chunk_filename, local_path)
        print(f"Saved video locally: {local_path}")
    except Exception as e:
        print(f"Local video save error for {chunk_filename}: {e}")

def save_to_firebase(detection_details):
    try:
        new_detection_ref = db_ref.push(detection_details)
        print(f"Detection stored in Firebase: {new_detection_ref.key}")
    except Exception as e:
        print(f"Firebase error: {e}")

def send_alert(detection_details):
    message_body = f"Shoplifting detected at {time.ctime()}! Probability: {detection_details['confidence']:.2f}"
    try:
        message = client.messages.create(
            body=message_body,
            from_=twilio_phone,
            to=security_phone
        )
        print(f"Alert sent: {message.sid}")
    except Exception as e:
        print(f"Alert error: {e}")

def trigger_led_blink():
    if ser is not None:
        try:
            ser.write(b'B')
            print("Sent blink command to Arduino")
        except serial.SerialException as e:
            print(f"Error sending command to Arduino: {e}")

chunk_writer = None
chunk_start_time = time.time()
chunk_filename = None
chunk_has_detection = False

while True:
    detection_details = None
    text_y_offset = 30

    ret, frame = cap.read()
    if not ret:
        print("End of video stream or connection lost. Attempting to reconnect...")
        cap.release()
        cap = initialize_video_source(video_source)
        if cap is None:
            print("Reconnection failed. Exiting.")
            break
        continue

    current_time = time.time()
    frame_count += 1

    if chunk_writer is None:
        chunk_filename = os.path.join("chunk_video", f"live_chunk_{int(current_time)}.mp4")
        chunk_writer = cv2.VideoWriter(chunk_filename, fourcc, fps, (frame_width, frame_height))
        chunk_start_time = current_time
        chunk_has_detection = False
        print(f"Started new chunk: {chunk_filename}")

    chunk_writer.write(frame)

    shoplifting_detected = False
    shoplifting_prob = 0.0
    detection_details = None

    if frame_count % frame_skip == 0:
        results = model.predict(frame, imgsz=480, conf=0.6, half=True, verbose=False)
        for result in results:
            for box in result.boxes:
                confidence = box.conf.item()
                class_id = int(box.cls.item())
                class_name = result.names[class_id]
                if class_name == "shoplifting" and confidence > confidence_threshold:
                    shoplifting_detected = True
                    shoplifting_prob = confidence
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    detection_details = {
                        'timestamp': datetime.now().isoformat(),
                        'confidence': float(confidence),
                        'coordinates': {'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2}
                    }
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label = f"Shoplifting: {confidence:.2f}"
                    cv2.putText(frame, label, (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
                    cv2.putText(frame, f"Action: Shoplifting, Prob: {confidence:.2f}",
                                (10, text_y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                    text_y_offset += 30
                    frame_filename = f"Detected_images/alert_frame_{int(current_time)}.jpg"
                    detection_details['local_frame_path'] = frame_filename
                    cv2.imwrite(frame_filename, frame)
                    print(f"Saved frame: {frame_filename}")
                    chunk_has_detection = True

    if current_time - chunk_start_time >= chunk_duration:
        chunk_writer.release()
        print(f"Completed chunk: {chunk_filename}, has_detection: {chunk_has_detection}")
        if chunk_has_detection:
            threading.Thread(target=save_local_video, args=(chunk_filename,)).start()
            threading.Thread(target=upload_to_cloudinary, args=(chunk_filename,)).start()
        else:
            if os.path.exists(chunk_filename):
                os.remove(chunk_filename)
                print(f"Deleted non-detected chunk: {chunk_filename}")
        chunk_writer = None
        chunk_filename = None

    if shoplifting_detected and (current_time - last_alert_time) > alert_cooldown:
        threading.Thread(target=send_alert, args=(detection_details,)).start()
        if detection_details:
            threading.Thread(target=save_to_firebase, args=(detection_details,)).start()
        threading.Thread(target=trigger_led_blink).start()
        last_alert_time = current_time

    status_label = "Shoplifting" if shoplifting_detected else "Normal"
    cv2.putText(frame, f"Status: {status_label}", (10, frame_height - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    cv2.imshow("Security Feed", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
if chunk_writer is not None:
    chunk_writer.release()
cap.release()
cv2.destroyAllWindows()
if ser is not None:
    ser.close()
    print("Closed Arduino serial connection")
print("Processing completed.")